# Домашнее задание 2.3: Связанные колебания и жёсткость

Динамика системы двух связанных масс описывается уравнениями:
$$ m \ddot{x}_1 = -k_0 x_1 - k_c(x_1 - x_2) - c(\dot{x}_1 - \dot{x}_2) $$
$$ m \ddot{x}_2 = -k_0 x_2 - k_c(x_2 - x_1) - c(\dot{x}_2 - \dot{x}_1) $$

Система сводится к векторному виду $\dot{\mathbf{u}} = \mathbf{A}\mathbf{u}$, где $\mathbf{u} = [x_1, x_2, v_1, v_2]^T$. Это позволяет использовать матричные операции для RK4 и получить аналитическое выражение шага для неявного метода трапеций:
$$ \mathbf{u}_{n+1} = \left(\mathbf{I} - \frac{\Delta t}{2}\mathbf{A}\right)^{-1} \left(\mathbf{I} + \frac{\Delta t}{2}\mathbf{A}\right) \mathbf{u}_n $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from scipy.signal import find_peaks

# Увеличиваем лимит памяти для анимации до 100 МБ
plt.rcParams['animation.embed_limit'] = 100.0

# Глобальная матрица системы
def get_A_matrix(m, k0, kc, c):
    return np.array([
        [0, 0, 1, 0],
        [0, 0, 0, 1],
        [-(k0+kc)/m, kc/m, -c/m, c/m],
        [kc/m, -(k0+kc)/m, c/m, -c/m]
    ])

def rk4_step(u, dt, A):
    k1 = A @ u
    k2 = A @ (u + 0.5 * dt * k1)
    k3 = A @ (u + 0.5 * dt * k2)
    k4 = A @ (u + dt * k3)
    return u + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

def verlet_step(x, v, a, dt, m, k0, kc):
    x_next = x + v * dt + 0.5 * a * dt**2
    a_next = np.array([
        (-k0*x_next[0] - kc*(x_next[0] - x_next[1])) / m,
        (-k0*x_next[1] - kc*(x_next[1] - x_next[0])) / m
    ])
    v_next = v + 0.5 * (a + a_next) * dt
    return x_next, v_next, a_next

def implicit_trapezoidal_step(u, dt, A):
    I = np.eye(4)
    M_inv = np.linalg.inv(I - 0.5 * dt * A)
    M_fwd = I + 0.5 * dt * A
    return (M_inv @ M_fwd) @ u

# Генерация координат пружины для анимации (зигзаг)
def get_spring(x_start, x_end, n_coils=10, width=0.3):
    length = x_end - x_start
    x = np.linspace(x_start, x_end, n_coils)
    y = np.zeros(n_coils)
    y[1:-1:2] = width
    y[2:-1:2] = -width
    return x, y

## 🟢 Базовый уровень: Слабая связь и биения

Слабая связь ($k_c \ll k_0$) без демпфирования ($c=0$). 
Аналитические частоты: $\omega_1 = \sqrt{k_0/m}$ (синфазная) и $\omega_2 = \sqrt{(k_0+2k_c)/m}$ (противофазная).
Период биений: $T_{beat} = 2\pi / (\omega_2 - \omega_1)$.

In [ ]:
m, k0, kc, c = 1.0, 10.0, 0.5, 0.0
A_weak = get_A_matrix(m, k0, kc, c)

# Аналитические частоты
w1 = np.sqrt(k0 / m)
w2 = np.sqrt((k0 + 2*kc) / m)
T_beat_analytic = 2 * np.pi / (w2 - w1)

def simulate_weak(dt):
    t_max = 80.0
    n_steps = int(t_max / dt)
    u_rk4 = np.zeros((n_steps + 1, 4))
    u_ver = np.zeros((n_steps + 1, 4))
    
    u0 = np.array([1.0, 0.0, 0.0, 0.0]) # Возбуждение первой массы
    u_rk4[0], u_ver[0] = u0, u0
    
    # Верле инициализация
    x_v, v_v = u0[:2], u0[2:]
    a_v = np.array([(-k0*x_v[0] - kc*(x_v[0] - x_v[1]))/m, (-k0*x_v[1] - kc*(x_v[1] - x_v[0]))/m])
    
    for i in range(n_steps):
        u_rk4[i+1] = rk4_step(u_rk4[i], dt, A_weak)
        x_v, v_v, a_v = verlet_step(x_v, v_v, a_v, dt, m, k0, kc)
        u_ver[i+1] = np.concatenate((x_v, v_v))
        
    t_arr = np.linspace(0, t_max, n_steps + 1)
    return t_arr, u_rk4, u_ver

# Моделирование с двумя разными шагами
dt1, dt2 = 0.01, 0.1
t1, rk4_1, ver_1 = simulate_weak(dt1)
t2, rk4_2, ver_2 = simulate_weak(dt2)

# Поиск численного периода биений
peaks, _ = find_peaks(rk4_1[:, 0], distance=int(10/dt1))
if len(peaks) >= 2:
    T_beat_num = t1[peaks[1]] - t1[peaks[0]]
else:
    T_beat_num = 0

print(f"Аналитический период биений: {T_beat_analytic:.3f} с")
print(f"Численный период биений (dt={dt1}): {T_beat_num:.3f} с")

# Расчет энергий для dt=0.01 (Verlet)
Ek1 = 0.5 * m * ver_1[:, 2]**2
Ek2 = 0.5 * m * ver_1[:, 3]**2
Ep = 0.5 * k0 * ver_1[:, 0]**2 + 0.5 * k0 * ver_1[:, 1]**2 + 0.5 * kc * (ver_1[:, 0] - ver_1[:, 1])**2
E_total = Ek1 + Ek2 + Ep

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Смещение (сравнение шагов)
ax1.plot(t1, rk4_1[:, 0], 'b-', alpha=0.7, label=f'x1 RK4 (dt={dt1})')
ax1.plot(t1, rk4_1[:, 1], 'r-', alpha=0.7, label=f'x2 RK4 (dt={dt1})')
ax1.plot(t2, ver_2[:, 0], 'c--', label=f'x1 Verlet (dt={dt2})')
ax1.set_title('Биения при слабой связи (RK4 и Верле)')
ax1.set_ylabel('Смещение')
ax1.legend(loc='upper right')

# Энергия
ax2.plot(t1, Ek1, 'b-', alpha=0.6, label='Кинетическая масса 1')
ax2.plot(t1, Ek2, 'r-', alpha=0.6, label='Кинетическая масса 2')
ax2.plot(t1, E_total, 'k-', lw=2, label='Полная механическая энергия')
ax2.set_title(f'Перекачка энергий (Метод Верле, dt={dt1})')
ax2.set_xlabel('Время, с')
ax2.set_ylabel('Энергия')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

## 🟡 Продвинутый уровень: Жёсткость и неявные методы

Увеличиваем жёсткость соединительной пружины ($k_c \gg k_0$) и добавляем вязкий демпфер ($c > 0$).
Возникает быстрая (относительное смещение $x_1 - x_2$) и медленная (совместные колебания) моды.

**Ответ на контрольный вопрос:** 
Да, можно. После затухания быстрого процесса неявный метод Адамса-Моултона (метод трапеций) позволяет значительно увеличить шаг интегрирования $\Delta t$, ориентируясь только на характерное время медленного движения. Явные методы (RK4) не могут этого сделать: если нарушить условие устойчивости $\Delta t < \frac{C}{|\lambda_{max}|}$, RK4 "взорвется", даже если быстрый процесс физически уже прекратился. Неявный метод абсолютно устойчив.

In [ ]:
kc_s, c_s, k0_s = 100.0, 5.0, 2.0
A_stiff = get_A_matrix(m, k0_s, kc_s, c_s)

def simulate_stiff(dt, t_max):
    n_steps = int(t_max / dt)
    u_rk = np.zeros((n_steps + 1, 4))
    u_imp = np.zeros((n_steps + 1, 4))
    u0 = np.array([2.0, -1.0, 0.0, 0.0]) # Возбуждаем обе моды
    u_rk[0], u_imp[0] = u0, u0
    
    for i in range(n_steps):
        u_imp[i+1] = implicit_trapezoidal_step(u_imp[i], dt, A_stiff)
        try:
            u_rk[i+1] = rk4_step(u_rk[i], dt, A_stiff)
        except OverflowError:
            u_rk[i+1] = np.nan
            
    return np.linspace(0, t_max, n_steps + 1), u_rk, u_imp

# 1. Мелкий шаг (оба метода стабильны)
t_fine, rk_fine, imp_fine = simulate_stiff(0.01, 20.0)

# 2. Крупный шаг (RK4 взорвется из-за жесткости)
t_coarse, rk_coarse, imp_coarse = simulate_stiff(0.12, 20.0)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Переходный процесс (мелкий шаг)
axes[0,0].plot(t_fine[t_fine<2], rk_fine[t_fine<2, 0], label='x1 (RK4, dt=0.01)')
axes[0,0].plot(t_fine[t_fine<2], rk_fine[t_fine<2, 1], label='x2 (RK4, dt=0.01)')
axes[0,0].set_title('Начальный переходный процесс')
axes[0,0].legend()

# Медленное движение (Неявный метод) - мелкий и крупный шаг
axes[0,1].plot(t_fine, imp_fine[:, 0], 'k-', alpha=0.5, label='x1 (Implicit, dt=0.01)')
axes[0,1].plot(t_coarse, imp_coarse[:, 0], 'r--', lw=2, label='x1 (Implicit, dt=0.12)')
axes[0,1].set_title('Точность неявного метода на медленной моде')
axes[0,1].legend()

# Медленное движение (RK4) - крупный шаг
axes[1,0].plot(t_coarse, rk_coarse[:, 0], 'g-', label='x1 (RK4, dt=0.12)')
axes[1,0].set_ylim(-3, 3)
axes[1,0].set_title('Численная неустойчивость RK4 (взрыв)')
axes[1,0].legend()

# Относительное смещение (x1 - x2) на мелком шаге
axes[1,1].plot(t_fine, imp_fine[:, 0] - imp_fine[:, 1], 'purple', label='x1 - x2')
axes[1,1].set_title('Быстрое затухание относительного движения')
axes[1,1].legend()

plt.tight_layout()
plt.show()

### Анимации систем (слабая и жесткая связь)
Отрисовка движения масс и соединительных пружин.
Для оптимизации размера встроенной анимации применяется прореживание кадров.

In [ ]:
fig_anim1, ax_a1 = plt.subplots(figsize=(10, 3))
ax_a1.set_xlim(-4, 4)
ax_a1.set_ylim(-1, 1)
ax_a1.set_title("Слабая связь: Биения (пружины отрисованы)")
ax_a1.get_yaxis().set_visible(False)

m1_w, = ax_a1.plot([], [], 'bo', ms=20, zorder=5)
m2_w, = ax_a1.plot([], [], 'ro', ms=20, zorder=5)
spr_left_w, = ax_a1.plot([], [], 'k-', lw=1.5)
spr_mid_w, = ax_a1.plot([], [], 'g-', lw=2.0)
spr_right_w, = ax_a1.plot([], [], 'k-', lw=1.5)

plt.close(fig_anim1)

# Прореживаем кадры для снижения веса анимации (отрисовка каждые 0.1 сек)
frame_step = int(0.1 / dt1) 
frames_w = len(t1) // frame_step

def update_anim1(frame):
    idx = frame * frame_step
    x1, x2 = rk4_1[idx, 0] - 1.5, rk4_1[idx, 1] + 1.5 # Базовые позиции -1.5 и 1.5
    
    m1_w.set_data([x1], [0])
    m2_w.set_data([x2], [0])
    
    xl, yl = get_spring(-4, x1)
    xm, ym = get_spring(x1, x2, n_coils=15)
    xr, yr = get_spring(x2, 4)
    
    spr_left_w.set_data(xl, yl)
    spr_mid_w.set_data(xm, ym)
    spr_right_w.set_data(xr, yr)
    
    return m1_w, m2_w, spr_left_w, spr_mid_w, spr_right_w

ani1 = animation.FuncAnimation(fig_anim1, update_anim1, frames=frames_w, interval=50, blit=True)
HTML(ani1.to_jshtml())

In [ ]:
# Анимация 2: Жёсткая связь + график относительного смещения
fig_anim2, (ax_a2, ax_graph) = plt.subplots(2, 1, figsize=(10, 6))

ax_a2.set_xlim(-4, 4)
ax_a2.set_ylim(-1, 1)
ax_a2.set_title("Сильная связь + демпфер: Синфазное движение")
ax_a2.get_yaxis().set_visible(False)

m1_s, = ax_a2.plot([], [], 'bo', ms=20, zorder=5)
m2_s, = ax_a2.plot([], [], 'ro', ms=20, zorder=5)
spr_left_s, = ax_a2.plot([], [], 'k-', lw=1.5)
spr_mid_s, = ax_a2.plot([], [], 'g-', lw=4.0) # Толстая пружина = жесткая связь
spr_right_s, = ax_a2.plot([], [], 'k-', lw=1.5)

ax_graph.set_xlim(0, 20)
ax_graph.set_ylim(-4, 4)
ax_graph.set_title("Относительное смещение (x1 - x2)")
ax_graph.set_xlabel("Время, с")
graph_rel, = ax_graph.plot([], [], 'purple', lw=2)

plt.tight_layout()
plt.close(fig_anim2)

frames_s = len(t_fine) // frame_step

def update_anim2(frame):
    idx = frame * frame_step
    x1, x2 = imp_fine[idx, 0] - 1.5, imp_fine[idx, 1] + 1.5
    
    m1_s.set_data([x1], [0])
    m2_s.set_data([x2], [0])
    
    xl, yl = get_spring(-4, x1)
    xm, ym = get_spring(x1, x2, n_coils=5, width=0.15)
    xr, yr = get_spring(x2, 4)
    
    spr_left_s.set_data(xl, yl)
    spr_mid_s.set_data(xm, ym)
    spr_right_s.set_data(xr, yr)
    
    graph_rel.set_data(t_fine[:idx+1], imp_fine[:idx+1, 0] - imp_fine[:idx+1, 1])
    
    return m1_s, m2_s, spr_left_s, spr_mid_s, spr_right_s, graph_rel

ani2 = animation.FuncAnimation(fig_anim2, update_anim2, frames=frames_s, interval=50, blit=True)
HTML(ani2.to_jshtml())